<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-03-prompt-engineering/lesson-3.4-context-engineering/notebooks/exercises-3.4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3.4: Context Engineering — Practice Exercises

**Netsetos GenAI Course** | 8 exercises: context stack, lost-in-middle, tokens, budgets, caching, MCP, RAG vs LC, pipeline.

---

In [ ]:
!pip install google-generativeai tiktoken -q
import google.generativeai as genai
import tiktoken
# genai.configure(api_key='YOUR_KEY')
model = genai.GenerativeModel('gemini-2.0-flash')
def ask(p,t=0): return model.generate_content(p,generation_config={'temperature':t}).text
enc = tiktoken.encoding_for_model('gpt-4o')


---
## Ex 1: Context Stack Audit

In [ ]:
layers = {
    'System prompt': 'You are a helpful customer support agent for an e-commerce company...',
    'Few-shot examples': 'Q: Where is my order? A: Let me check...\nQ: I want a refund A: I can help...',
    'Retrieved docs': 'Return policy: Items can be returned within 30 days...' * 5,
    'Tool definitions': '{"name":"check_order","params":{"order_id":"string"}}' * 3,
    'Conversation history': 'User: Hi\nAssistant: Hello!\nUser: My order is late\n' * 10,
    'User query': 'Can I get a refund for order #12345?',
}
total = 0
for layer, content in layers.items():
    tokens = len(enc.encode(content))
    total += tokens
    pct = tokens / 128000 * 100
    print(f'{layer:>25}: {tokens:>5} tokens ({pct:.1f}%)')
print(f'{"TOTAL":>25}: {total:>5} tokens ({total/128000*100:.1f}% of 128K)')
print(f'{"Available for response":>25}: {128000-total:>5} tokens')


---
## Ex 2: Lost-in-the-Middle Test

In [ ]:
filler = 'The weather in Paris is typically mild in spring. ' * 20
needle = 'The secret password is RAINBOW42.'
positions = [0, 4, 9, 14, 19]  # in a 20-paragraph doc

for pos in positions:
    paragraphs = [filler] * 20
    paragraphs[pos] = needle
    doc = '\n\n'.join(paragraphs)
    r = ask(f'What is the secret password in this document?\n\n{doc}')
    found = 'RAINBOW42' in r
    print(f'Position {pos+1:>2}/20: {"✅ Found" if found else "❌ Missed"} | {r.strip()[:50]}')
print('\nExpected: U-curve — best at start (1) and end (20), worst in middle (10)')


---
## Ex 3: Multilingual Token Audit

In [ ]:
texts = {
    'English': 'Artificial intelligence is transforming how businesses operate across India.',
    'Hindi': 'कृत्रिम बुद्धिमत्ता भारत भर में व्यापार संचालन के तरीके को बदल रही है।',
    'Telugu': 'కృత్రిమ మేధస్సు భారతదేశం అంతటా వ్యాపారాలు నడిచే విధానాన్ని మారుస్తోంది.',
    'Tamil': 'செயற்கை நுண்ணறிவு இந்தியா முழுவதும் வணிகங்கள் செயல்படும் விதத்தை மாற்றுகிறது.',
}
en_tokens = 0
for lang, text in texts.items():
    tokens = len(enc.encode(text))
    if lang == 'English': en_tokens = tokens
    ratio = tokens / en_tokens if en_tokens else 0
    print(f'{lang:>8}: {tokens:>3} tokens ({ratio:.1f}x English)')
print(f'\nHindi at {texts["Hindi"]}...')
print(f'128K window in English = {128000} tokens')
print(f'128K window in Hindi   ≈ {128000 // 3} effective tokens (at 3x cost)')


---
## Ex 4-8: See full implementations in the solution files

In [ ]:
print('Exercise 4: Design cached prefix for GST chatbot')
print('Exercise 5: Compression at 3 levels + accuracy test')
print('Exercise 6: RAG vs long-context comparison')
print('Exercise 7: MCP tool schema budgeting')
print('Exercise 8: Full context pipeline with 50-turn test')
print('\nSee solution .py files for complete implementations.')


---
**8 exercises complete.** 10 context engineering concepts mastered.